# 06 · Which chips are seasonally stable, and why does anything shift at all?

Entry 4's fire study found something odd: chips with essentially zero burn severity (dNBR near
zero) still showed a large embedding shift between two dates -- averaging roughly 0.6-0.9
(1 minus cosine similarity) even with no fire involved. Two explanations are consistent with
that: either real seasonal content genuinely changes that much between two arbitrary dates
(snow, dormancy, crop cycles), or Clay's explicit time-conditioning input shifts the embedding
by date somewhat independent of what's actually in the pixels. This notebook tests both ideas
directly:

1. Fetch a winter mosaic over the **same 725-chip AOI and grid** as the main study, and embed it
   with Clay -- this notebook reuses `docs/data/embeddings.bin` as the summer baseline directly,
   no need to re-embed anything from notebooks 01-03.
2. For every chip, measure embedding shift (summer 2024 -> winter 2023/24) and check which
   clusters/land-cover types are seasonally **stable** (small shift) vs. **volatile** (large
   shift) -- correlated against the cluster, NDVI, and elevation/slope data already committed
   from earlier entries.
3. **The ablation entry 4 called for**: re-embed a sample of winter chips twice -- once with
   their real winter date, once with the *same pixels* but a fake summer date fed to the
   metadata -- to isolate how much of the shift is pixel content vs. date-conditioning alone.

**Requires a GPU runtime.** Self-contained: reads the summer baseline straight from committed
`docs/data/` files, no Drive mount needed.

In [ ]:
REPO_URL = "https://github.com/ZanderHirman08/SATEMB.git"

import os

if not os.path.exists("SATEMB"):
    !git clone {REPO_URL}
%cd SATEMB
!pip install -q -r environment/requirements-colab.txt

In [ ]:
import sys

sys.path.append(os.getcwd())

import json

import matplotlib.pyplot as plt
import numpy as np
import odc.stac
import pandas as pd
import torch
from scipy import stats

from src import clay_embed, stac_utils, viz_utils

assert torch.cuda.is_available(), "No GPU detected -- switch runtime type to T4 GPU and re-run"
device = "cuda"
os.makedirs("docs/figures", exist_ok=True)

with open("docs/data/chips.geojson") as f:
    chips_geojson = json.load(f)
features = chips_geojson["features"]

with open("docs/data/embeddings.bin", "rb") as f:
    flat = np.frombuffer(f.read(), dtype="<f4")
emb_summer = flat.reshape(len(features), -1)

print(f"Loaded {len(features)} chips, {emb_summer.shape[1]}-dim summer baseline embeddings")

## Fetch a winter mosaic over the same AOI

Same bbox/CRS/resolution as notebook 01 (so the pixel grid, and therefore chip ids, line up
exactly), same per-tile least-cloudy selection since this AOI spans two Sentinel-2 tiles. Unlike
notebook 05, snow is left in on purpose here -- it's part of the real winter state we're trying
to measure, not something to filter out.

In [ ]:
catalog = stac_utils.open_catalog()
WINTER_RANGE = "2023-12-01/2024-02-28"

winter_items_all = stac_utils.search_sentinel2(catalog, datetime_range=WINTER_RANGE, max_cloud_cover=30.0)
print(f"{len(winter_items_all)} candidate winter scenes")
winter_items = stac_utils.select_least_cloudy_per_tile(winter_items_all)

winter_ds = odc.stac.load(
    winter_items, bands=stac_utils.S2_BANDS, bbox=stac_utils.FRONT_RANGE_BBOX,
    crs="EPSG:32613", resolution=stac_utils.GSD_M, groupby="solar_day",
    chunks={"x": 1024, "y": 1024},
)
winter_mosaic = winter_ds.to_array(dim="band").median(dim="time").compute()
print(winter_mosaic.shape, winter_mosaic.dtype)

In [ ]:
rgb = winter_mosaic.sel(band=["B04", "B03", "B02"]).values.astype("float32")
out = np.zeros_like(rgb)
for i in range(3):
    lo, hi = np.nanpercentile(rgb[i], [2, 98])
    out[i] = np.clip((rgb[i] - lo) / (hi - lo + 1e-6), 0, 1)

plt.figure(figsize=(10, 10))
plt.imshow(out.transpose(1, 2, 0))
plt.title(f"Winter mosaic ({WINTER_RANGE}) -- confirm real winter conditions (snow, dormancy)")
plt.axis("off")
plt.savefig("docs/figures/seasonal_winter_true_color.png", dpi=150, bbox_inches="tight")
plt.show()

## Match winter pixels to the existing chip ids

In [ ]:
def centroid_of(feature):
    coords = feature["geometry"]["coordinates"][0]
    lons = [c[0] for c in coords]
    lats = [c[1] for c in coords]
    return (min(lats) + max(lats)) / 2, (min(lons) + max(lons)) / 2

height, width = winter_mosaic.sizes["y"], winter_mosaic.sizes["x"]
grid_by_id = {c["id"]: c for c in stac_utils.make_pixel_chip_grid(height, width)}

NODATA_FRAC_THRESHOLD = 0.05
matched_idx, winter_pixels, lats, lons = [], [], [], []
for i, f in enumerate(features):
    win = grid_by_id.get(f["properties"]["id"])
    if win is None:
        continue
    patch = winter_mosaic.values[:, win["y_slice"], win["x_slice"]]
    if np.isnan(patch).mean() > NODATA_FRAC_THRESHOLD:
        continue
    lat, lon = centroid_of(f)
    matched_idx.append(i)
    winter_pixels.append(np.nan_to_num(patch, nan=0.0).astype("float32"))
    lats.append(lat)
    lons.append(lon)

winter_pixels = np.stack(winter_pixels)
matched_idx = np.array(matched_idx)
print(f"Matched {len(matched_idx)}/{len(features)} chips between the summer baseline and this winter mosaic")

## Embed winter chips with Clay

In [ ]:
ckpt_path = clay_embed.download_checkpoint()
metadata_path = clay_embed.download_metadata_yaml()
model = clay_embed.load_model(ckpt_path, metadata_path, device=device)
wavelengths, band_means, band_stds = clay_embed.load_band_stats(metadata_path)

WINTER_DATE = pd.Timestamp("2024-01-15")  # representative date within WINTER_RANGE

def embed_batch(pixels, date, lats, lons, batch_size=16):
    # Smaller default batch size than notebooks 01/02/05 used, plus an
    # explicit cache clear after every batch -- this notebook calls
    # embed_batch twice in one session (main pass + ablation pass right
    # after), and a free-tier T4's ~15GB can run out if PyTorch's caching
    # allocator holds onto memory between the two calls. If you still hit
    # "CUDA out of memory", Runtime -> Restart session and re-run from the
    # top (clears any leftover GPU memory from earlier notebooks/attempts
    # in the same runtime), or lower batch_size further, e.g. embed_batch(..., batch_size=8).
    dates = [date] * len(lats)
    out = []
    for start in range(0, len(lats), batch_size):
        end = min(start + batch_size, len(lats))
        batch_pixels = clay_embed.normalize_chips(pixels[start:end], band_means, band_stds)
        time_feats, latlon_feats = clay_embed.make_time_latlon_tensors(
            dates[start:end], lats[start:end], lons[start:end]
        )
        batch_emb = clay_embed.encode_batch(model, batch_pixels, time_feats, latlon_feats, wavelengths, device=device)
        out.append(batch_emb)
        del batch_pixels, time_feats, latlon_feats
        torch.cuda.empty_cache()
    return np.concatenate(out, axis=0)

emb_winter = embed_batch(winter_pixels, WINTER_DATE, lats, lons)
print(f"Embedded {len(matched_idx)} winter chips, dim={emb_winter.shape[1]}")

## Seasonal shift: which chips move, which stay put?

In [ ]:
def cosine_sim_rows(a, b):
    a_n = a / np.linalg.norm(a, axis=1, keepdims=True)
    b_n = b / np.linalg.norm(b, axis=1, keepdims=True)
    return (a_n * b_n).sum(axis=1)

emb_summer_matched = emb_summer[matched_idx]
shift = 1 - cosine_sim_rows(emb_summer_matched, emb_winter)

matched_features = [features[i] for i in matched_idx]
clusters = np.array([f["properties"]["cluster"] for f in matched_features])
ndvi = np.array([f["properties"]["ndvi"] for f in matched_features])
has_elev = "elevation" in matched_features[0]["properties"]
elevation = np.array([f["properties"].get("elevation", np.nan) for f in matched_features]) if has_elev else None

print(f"Seasonal shift: mean={shift.mean():.3f}, std={shift.std():.3f}, min={shift.min():.3f}, max={shift.max():.3f}")
print(f"\nCompare to entry 4's fire-study baseline shift (~0.6-0.9 for a ~3-month Dec->Mar gap) --")
print("this is a ~6-8 month summer->winter gap over the full 725-chip AOI, a stronger seasonal test.")

In [ ]:
unique_clusters = sorted(set(clusters.tolist()))
cluster_means = [shift[clusters == c].mean() for c in unique_clusters]
cluster_stds = [shift[clusters == c].std() for c in unique_clusters]

plt.figure(figsize=(8, 5))
plt.bar([str(c) for c in unique_clusters], cluster_means, yerr=cluster_stds, capsize=4, color="#f58231")
plt.xlabel("embedding cluster")
plt.ylabel("mean seasonal shift (1 - cosine similarity), \u00b1 std")
plt.title("Summer -> winter embedding shift per cluster")
plt.tight_layout()
plt.savefig("docs/figures/seasonal_shift_by_cluster.png", dpi=150)
plt.show()

for c, m, s in zip(unique_clusters, cluster_means, cluster_stds):
    print(f"cluster {c}: shift={m:.3f} (std {s:.3f}, n={int((clusters == c).sum())})")

In [ ]:
r_ndvi = stats.pearsonr(ndvi, shift)
print(f"Seasonal shift vs. NDVI: r={r_ndvi[0]:.3f}, p={r_ndvi[1]:.4f}")

if elevation is not None and not np.all(np.isnan(elevation)):
    valid = ~np.isnan(elevation)
    r_elev = stats.pearsonr(elevation[valid], shift[valid])
    print(f"Seasonal shift vs. elevation: r={r_elev[0]:.3f}, p={r_elev[1]:.4f}")
    print("(A positive r here would suggest higher/snowier chips are more seasonally volatile.)")

fig, axes = plt.subplots(1, 2 if elevation is not None else 1, figsize=(13 if elevation is not None else 7, 5.5))
axes = axes if elevation is not None else [axes]
axes[0].scatter(ndvi, shift, s=12, alpha=0.6, color="#5ec8ff")
axes[0].set_xlabel("NDVI")
axes[0].set_ylabel("seasonal shift")
axes[0].set_title(f"vs. NDVI (r={r_ndvi[0]:.2f})")
if elevation is not None:
    axes[1].scatter(elevation[valid], shift[valid], s=12, alpha=0.6, color="#4fd0c4")
    axes[1].set_xlabel("elevation (m)")
    axes[1].set_ylabel("seasonal shift")
    axes[1].set_title(f"vs. elevation (r={r_elev[0]:.2f})")
plt.tight_layout()
plt.savefig("docs/figures/seasonal_shift_correlations.png", dpi=150)
plt.show()

In [ ]:
rows = [int(f["properties"]["id"][1:4]) for f in matched_features]
cols = [int(f["properties"]["id"][5:8]) for f in matched_features]
n_rows, n_cols = max(rows) + 1, max(cols) + 1

shift_grid = np.full((n_rows, n_cols), np.nan)
for r, c, val in zip(rows, cols, shift):
    shift_grid[r, c] = val

plt.figure(figsize=(10, 10))
im = plt.imshow(shift_grid, cmap="inferno")
plt.colorbar(im, label="seasonal shift (1 - cosine similarity)")
plt.title("Seasonal stability map -- bright = volatile, dark = stable across the year")
plt.axis("off")
plt.savefig("docs/figures/seasonal_stability_map.png", dpi=150, bbox_inches="tight")
plt.show()

## The ablation: how much of the shift is just the date, not the pixels?

Take a sample of winter chips and embed them **twice with identical pixels**: once with their
real winter date, once with a fake summer date substituted into the metadata. If Clay's
time-conditioning alone moves the embedding by roughly as much as the real summer-vs-winter
shift, that means most of what looked like "seasonal content change" above is actually just the
model reacting to the date label. If the date-only shift is small, the real content change
(snow, dormancy) is doing the work.

In [ ]:
SAMPLE_N = 30
rng = np.random.default_rng(0)
sample = rng.choice(len(matched_idx), size=min(SAMPLE_N, len(matched_idx)), replace=False)

FAKE_SUMMER_DATE = pd.Timestamp("2024-07-15")
sample_lats = [lats[i] for i in sample]
sample_lons = [lons[i] for i in sample]

emb_winter_fakesummer = embed_batch(winter_pixels[sample], FAKE_SUMMER_DATE, sample_lats, sample_lons)

shift_date_only = 1 - cosine_sim_rows(emb_winter[sample], emb_winter_fakesummer)
shift_real_total = shift[sample]

print(f"Date-metadata-only shift (same winter pixels, fake summer date):  mean={shift_date_only.mean():.3f}, std={shift_date_only.std():.3f}")
print(f"Real total shift (real summer embedding vs. real winter, same {len(sample)} chips): mean={shift_real_total.mean():.3f}, std={shift_real_total.std():.3f}")
print(f"\nDate-only shift is {100 * shift_date_only.mean() / shift_real_total.mean():.0f}% of the real total shift's magnitude.")
print("A high percentage means the date label itself, not the actual pixel content, is doing")
print("most of the work -- worth knowing before reading any 'embedding shift' as pure content change.")

In [ ]:
plt.figure(figsize=(7, 5.5))
plt.scatter(shift_real_total, shift_date_only, s=30, alpha=0.7, color="#f032e6")
lims = [0, max(shift_real_total.max(), shift_date_only.max()) * 1.05]
plt.plot(lims, lims, "--", color="gray", linewidth=1, label="y = x")
plt.xlabel("real total shift (real pixels + real date)")
plt.ylabel("date-only shift (winter pixels + fake summer date)")
plt.title("How much of the real shift does date-swapping alone explain?")
plt.legend()
plt.tight_layout()
plt.savefig("docs/figures/seasonal_ablation.png", dpi=150)
plt.show()

## Save seasonal shift onto the live map's data

In [ ]:
extra_props = {
    matched_features[j]["properties"]["id"]: {"seasonal_shift": round(float(shift[j]), 4)}
    for j in range(len(matched_idx))
}
viz_utils.add_properties_to_geojson("docs/data/chips.geojson", extra_props)

print("\nDone. Commit docs/data/chips.geojson (updated) and the new docs/figures/seasonal_*.png files.")